# Structured output

Goal: stop parsing the model's prose and get back an object with fields. Describe the shape
with a pydantic model, hand it to the API, read attributes off the result.

## Setup

`.env` sits at the repo root, two levels up from here.

In [ ]:
import os
from dotenv import load_dotenv

# .env lives at the repo root, two levels up from this notebook
load_dotenv("../../.env")

key = os.environ.get("OPENAI_API_KEY")

The client reads `OPENAI_API_KEY` from the environment by itself, so there's nothing to pass.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

## Describe the shape

A pydantic model is the schema. Field names and types are what the model has to fill in.

In [ ]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

Under the hood it's a JSON schema - this is what actually gets sent to the API.

In [ ]:
CalendarEvent.model_json_schema()

## Ask for that shape

`responses.parse` with `text_format=` is the structured call: it sends the schema along and
validates the reply against it.

In [ ]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

`.parsed` on the content block is the `CalendarEvent` instance, already validated.

In [ ]:
response.output[0].content[0].parsed

`.text` is the raw JSON string the model produced, before pydantic touched it.

In [ ]:
response.output[0].content[0].text

## The short way to the object

`response.output_parsed` skips walking `output[0].content[0]` and hands back the same instance.

In [ ]:
event = response.output_parsed

In [ ]:
event

`output_text` still works on a parsed response - it's the JSON as text.

In [ ]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent
)
print(response.output_text)

## The same question with no schema

`responses.create` without `text_format` gets prose back. Readable, but nothing you can index
into, and the shape changes between runs.

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Two runs of the unstructured call, pasted in as a string so re-running the notebook doesn't
lose them. Same facts, different layout each time - that's the problem structured output solves.

In [ ]:
'''
Event Information:
- Event: Science Fair
- Participants: Alice and Bob
- Day: Friday


Event: Science Fair  
Participants: Alice and Bob  
Day: Friday
'''

# Structured RAG

Goal: reuse the RAG pipeline from the previous module, then make its answer structured.

## Index the docs

Same three steps as before: read the repo, chunk the documents, fit a keyword index over them.

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

## Search

In [ ]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

## Prompt

The search results go into the prompt as JSON, so the model sees the metadata too.

In [ ]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

## The plain, unstructured call

Returns `output_text` - a string.

In [ ]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

## Wire the pieces together

Search, build the prompt, ask the model.

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [ ]:
answer = rag('how do I implement LLM as a Judge?')

## The structured call

Same function, but `responses.parse` with a `text_format`, returning `output_parsed` instead
of text. The output type is a parameter, so one function serves every schema below.

In [ ]:
def llm_structured(
        user_prompt,
        output_type,
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

A quick check with the calendar model from the first half, to confirm it works.

In [ ]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [ ]:
response

## A schema for RAG answers

`found_answer` is the interesting field: it lets the caller tell "here's the answer" from
"it isn't in the docs" without reading the prose.

In [ ]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [ ]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

A question the docs do cover.

In [ ]:
answer = rag_structured('how do i do llm evals?')

print(answer.answer[:500])
print(answer.found_answer)

One they don't. The answer still comes back as a string - the model writes an apology into
`answer` rather than leaving it empty, because the schema says `answer` is required.

In [ ]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:500])
print(answer.found_answer)

Both fields are in `required`, which is why there's no way to omit `answer`.

In [ ]:
RAGResponse.model_json_schema()

## Making the answer optional

`Optional[str] = None` is the obvious fix.

In [ ]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

The schema now allows null for `answer`.

In [ ]:
RAGResponse.model_json_schema()

But the model still fills it in. Nothing told it when to use null - the type allows it,
the prompt doesn't ask for it.

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

Run it again and the behaviour isn't stable either.

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

## Ask for it in the instructions

One extra line in the system prompt, telling the model when `answer` should be None.

In [ ]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

## Ask for it in the docstring

Instructions back to the original, so the next attempt is a fair comparison.

In [ ]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

A model docstring travels with the schema as its `description`, so the rule sits next to the
field it governs instead of in the prompt.

In [ ]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

The docstring shows up as `description` on the schema.

In [ ]:
RAGResponse.model_json_schema()

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

## Ask for it field by field

`Field(description=...)` is the finest-grained version: each field carries its own instruction.

In [ ]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [ ]:
RAGResponse.model_json_schema()

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

## A richer response model

Beyond the answer: a confidence score, a category from a fixed set via `Literal`, and
follow-up questions. All of it in one call, all of it typed.

In [ ]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [ ]:
RAGResponse.model_json_schema()

A question the docs cover.

In [ ]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [ ]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

And one they don't - watch what happens to `confidence` and `answer_type`.

In [ ]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [ ]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

## Two shapes, one of which must hold

Nesting the two outcomes in separate models makes them mutually exclusive by construction, and
a `model_validator` enforces that exactly one is set. The docstring spells the rule out for the
model as well.

In [ ]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [ ]:
AnswerResponse.model_json_schema()

The question the docs don't cover.

In [ ]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

And one they do.

In [ ]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

The validator is a pydantic guarantee, not an API one: construct the model with neither field
set and it refuses, so a malformed reply fails here rather than downstream.

In [ ]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)